# PokouAI — Fine-tune Gemma 4 on Cocoa Diseases

**Goal**: QLoRA fine-tune Gemma 4 (multimodal) on ~50k augmented cocoa pod images across 6 classes and 4 languages, export GGUF Q4_K_M for on-device inference.

**Target device**: entry-level Android, 2–3 GB RAM → **E2B is the primary variant** (~1.5 GB quantized, fits with OS + camera headroom). E4B is trained as an optional 'premium' variant for phones with ≥4 GB RAM.

**Platform**: Kaggle (T4 x2, 30 GB VRAM combined).

**Stack**: Unsloth (2× faster, ~60% less VRAM) + PEFT + TRL.

## Why Unsloth
Unsloth rewrites the Gemma forward/backward pass in Triton, enabling QLoRA fine-tuning on a single free-tier T4. Without it, QLoRA + gradient checkpointing still OOMs at sequence length 2048 with images. Unsloth makes E4B run at ~12 GB peak (E2B at ~7 GB).

## Licensing
Gemma 4 weights are distributed publicly under the Gemma Terms of Use. The direct `google/gemma-4-*-it` repos load without gated access — no click-through required. The notebook tries the direct HF repo first and falls back to Unsloth's pre-quantized 4-bit repo only if that fails.

## 1 — Environment setup

In [ ]:
!pip install -qU "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qU transformers accelerate peft trl datasets bitsandbytes pillow

In [ ]:
import os, json, torch
from pathlib import Path
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

print('CUDA:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

## 2 — Load Gemma 4 (E2B primary, E4B optional)

`VARIANT` controls which size to train. Defaults to `e2b` for on-device deployment.

Strategy per variant:
1. **Try direct HF repo** (`google/gemma-4-{variant}-it`) — public, no license click-through.
2. **Fall back** to Unsloth's pre-quantized 4-bit repo if the direct load errors (network, version mismatch, etc.).

In [ ]:
VARIANT = 'e2b'  # 'e2b' (primary, ~2B params) or 'e4b' (premium, ~4B params)
MAX_SEQ_LEN = 2048

DIRECT_REPO = f'google/gemma-4-{VARIANT}-it'
FALLBACK_REPO = f'unsloth/gemma-4-{VARIANT}-it-bnb-4bit'

def load_model():
    try:
        print(f'trying direct load: {DIRECT_REPO}')
        return FastModel.from_pretrained(
            model_name=DIRECT_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )
    except Exception as e:
        print(f'direct load failed ({type(e).__name__}: {e}) — falling back to {FALLBACK_REPO}')
        return FastModel.from_pretrained(
            model_name=FALLBACK_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )

model, tokenizer = load_model()
print(f'loaded {VARIANT.upper()}:', type(model).__name__)

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    finetune_vision_layers=True,
    finetune_language_layers=True,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

## 3 — Load dataset

Expects the Kaggle dataset `pokou-ai-cocoa-training-data` mounted at `/kaggle/input/pokou-ai-cocoa-training-data/`.

In [ ]:
from datasets import load_dataset

DATA_ROOT = Path('/kaggle/input/pokou-ai-cocoa-training-data')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
VAL_JSONL = DATA_ROOT / 'val.jsonl'

train_ds = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
val_ds = load_dataset('json', data_files=str(VAL_JSONL), split='train')

print('train:', len(train_ds), '| val:', len(val_ds))
print('first example keys:', train_ds[0].keys())

In [ ]:
# Format into Gemma multimodal chat template
from PIL import Image

def to_chat(example):
    messages = example['messages']
    user_content = messages[0]['content']
    assistant_content = messages[1]['content']
    image_rel = next(c['path'] for c in user_content if c['type'] == 'image')
    image_path = str(DATA_ROOT / image_rel)
    user_text = next(c['text'] for c in user_content if c['type'] == 'text')
    assistant_text = assistant_content[0]['text']
    return {
        'image': Image.open(image_path).convert('RGB'),
        'conversations': [
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': user_text}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
        ],
    }

train_ds = train_ds.map(to_chat)
val_ds = val_ds.map(to_chat)

## 4 — Training config

In [ ]:
OUTPUT_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}'

# E2B runs the larger batch comfortably on a single T4; E4B needs smaller per-device batch
PER_DEVICE_BATCH = 4 if VARIANT == 'e2b' else 2
GRAD_ACCUM = 2 if VARIANT == 'e2b' else 4

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='epoch',
        save_total_limit=3,
        bf16=True,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='conversations',
        report_to='none',
        seed=42,
    ),
)

## 5 — Train

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

## 6 — Save LoRA adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('LoRA adapter saved to', OUTPUT_DIR)

## 7 — Export to GGUF Q4_K_M for on-device inference

Size targets:
- **E2B Q4_K_M**: ~1.3–1.6 GB file, runs in ~2 GB RAM on llama.cpp (fits 2–3 GB phones)
- **E4B Q4_K_M**: ~2.5–3.0 GB file, needs ~4 GB RAM (premium phones only)

In [ ]:
GGUF_OUTPUT = f'/kaggle/working/cocoa_v1_{VARIANT}_gguf'
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method='q4_k_m')
!ls -lh {GGUF_OUTPUT}/

## 8 — Quick sanity check (single-image inference)

In [ ]:
from transformers import TextStreamer

sample = val_ds[0]
inputs = tokenizer.apply_chat_template(
    sample['conversations'][:1],
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
).to('cuda')

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, max_new_tokens=400, streamer=streamer, do_sample=False)

## 9 — Train the other variant

To produce both models for the submission, restart the kernel, set `VARIANT = 'e4b'` in cell 2, and re-run from Section 3. Outputs land in a separate folder (`cocoa_v1_e4b_gguf/`), so nothing is overwritten.